# Django ORM Tools

## Advanced Query Filters

```python
from shop.models import Product

# Values in a list
Product.objects.filter(category_id__in=[1, 2, 3])

# NULL check
Product.objects.filter(description__isnull=True)

# Comparison
Product.objects.filter(price__gt=100)

# Combining filters
Product.objects.filter(price__gt=100).exclude(stock=0).first()
```


## Custom Model Managers

You can add custom query methods to a model by extending `Manager`:

```python
class PublishedManager(models.Manager):
    def get_queryset(self):
        return super().get_queryset().filter(status='published')

    def by_author(self, author):
        return self.get_queryset().filter(author=author)

class Article(models.Model):
    title = models.CharField(max_length=200)
    status = models.CharField(max_length=20)
    author = models.ForeignKey('Author', on_delete=models.CASCADE)

    objects = models.Manager()        # default manager kept
    published = PublishedManager()    # custom manager
```

Usage:
```python
Article.published.all()
Article.published.by_author(some_author)
```


## Aggregation

`aggregate()` computes a single summary value across the entire queryset and returns a dict:

```python
from django.db.models import Count, Sum, Avg
from shop.models import Order

Order.objects.aggregate(
    total_orders=Count('id'),
    total_revenue=Sum('amount'),
    avg_order=Avg('amount'),
)
# → {'total_orders': 42, 'total_revenue': Decimal('8400.00'), 'avg_order': ...}
```


## Annotation

`annotate()` adds a computed column to each row in the queryset:

```python
from django.db.models import Count

from blog.models import Author

# Each author object now has a `post_count` attribute
authors = Author.objects.annotate(post_count=Count('posts'))
for author in authors:
    print(author.name, author.post_count)
```

Annotations can be used in `filter()` and `order_by()`:
```python
Author.objects.annotate(post_count=Count('posts')).filter(post_count__gte=5)
```


## Filtering with Q Objects

`Q` objects allow building complex queries with `&` (AND) and `|` (OR):

```python
from django.db.models import Q
from shop.models import Product

# Products that are cheap OR in stock
Product.objects.filter(Q(price__lt=10) | Q(stock__gt=0))

# Products that are cheap AND in stock
Product.objects.filter(Q(price__lt=10) & Q(stock__gt=0))

# NOT
Product.objects.filter(~Q(status='discontinued'))
```


## Cookies and Sessions (Introduction)

HTTP is a stateless protocol — the server does not remember previous requests. Django provides two mechanisms to store state:

- **Cookies** — small pieces of data stored on the client (browser).
- **Sessions** — server-side storage identified by a session ID sent via a cookie.

These are covered in detail in the next session.


## Summary

- Custom managers extend `models.Manager` to encapsulate reusable query logic.
- `aggregate()` returns a single dict with summary values (count, sum, avg).
- `annotate()` adds a per-row computed field that can be filtered or sorted.
- `Q` objects combine filter conditions using `|` (OR), `&` (AND), and `~` (NOT).
